In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%load_ext autoreload
%autoreload 2
from No_atten_drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

In [3]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data()

load nci


100%|██████████| 25/25 [00:02<00:00,  9.98it/s]


Done!


In [4]:
PATH = "../nci_data/"

In [13]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            (_, _, _, best_metrics, _, _, _) = drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [14]:
name = "NCI_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-27 18:26:21,652] Using an existing study with name 'NCI_GAT' instead of creating a new one.
[I 2025-03-27 18:26:22,339] Trial 9 pruned. Invalid layer size configuration


Using device: cuda


  0%|          | 0/5 [00:00<?, ?it/s]
[I 2025-03-27 18:26:24,876] Trial 10 pruned. OOM at trial 10


Pruned trial 10: CUDA OOM


[I 2025-03-27 18:26:25,520] Trial 11 pruned. Invalid layer size configuration


...

Best model found at epoch 5
Using device: cuda


100%|██████████| 5/5 [00:04<00:00,  1.24it/s]


Best model found at epoch 5
Using device: cuda


 40%|████      | 2/5 [00:02<00:03,  1.14s/it]
[W 2025-03-27 18:34:49,034] Trial 34 failed with parameters: {'dropout1': 0.4, 'dropout2': 0.2, 'hidden1': 512, 'hidden2': 256, 'hidden3': 128, 'heads': 2, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.005066750844834099, 'weight_decay': 1.5261414019838627e-06, 'scheduler': 'Plateau', 'gnn_layer': 'GAT', 'patience_plateau': 7, 'thresh_plateau': 0.0027354070247990357, 'amsgrad': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3533722/2879114162.py", line 89, in objective
    (_, _, _, best_metrics, _, _, _) = drGAT.train(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py", line 265, in train
    validate_model(
  File "/spin1/home/linux/inouey2/drGAT/drGAT/drGAT.py", line 444, in validate_model
    outputs, atten

KeyboardInterrupt: 

In [15]:
study.trials_dataframe()

,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,params_lr,params_momentum,params_nesterov,params_optimizer,params_patience_plateau,params_scheduler,params_step_size,params_thresh_plateau,params_weight_decay,state
0,0,NaN,NaN,NaN,NaN,2025-03-27 18:14:02.936825,2025-03-27 18:14:29.493193,0 days 00:00:26.556368,NaN,swish,...,0.000096,NaN,NaN,AdamW,NaN,Step,21.0,NaN,0.003508,FAIL
1,1,NaN,NaN,NaN,NaN,2025-03-27 18:14:29.528494,2025-03-27 18:14:29.964035,0 days 00:00:00.435541,40.0,gelu,...,0.000026,NaN,NaN,Adam,NaN,Cosine,NaN,NaN,0.000004,PRUNED
2,2,NaN,NaN,NaN,NaN,2025-03-27 18:14:30.002582,2025-03-27 18:14:54.645663,0 days 00:00:24.643081,41.0,relu,...,0.000990,NaN,NaN,Adam,NaN,Cosine,NaN,NaN,0.000078,FAIL
3,3,NaN,NaN,NaN,NaN,2025-03-27 18:14:54.680293,2025-03-27 18:15:18.601325,0 days 00:00:23.921032,NaN,relu,...,0.000088,NaN,NaN,Adam,NaN,None,NaN,NaN,0.000002,FAIL
4,4,NaN,NaN,NaN,NaN,2025-03-27 18:15:18.635639,2025-03-27 18:15:54.158974,0 days 00:00:35.523335,NaN,relu,...,0.000047,0.885344,True,SGD,NaN,Step,30.0,NaN,0.000001,FAIL
5,5,NaN,NaN,NaN,NaN,2025-03-27 18:15:54.192808,2025-03-27 18:16:11.910792,0 days 00:00:17.717984,37.0,gelu,...,0.001725,NaN,NaN,Adam,NaN,Cosine,NaN,NaN,0.000083,FAIL
6,6,NaN,NaN,NaN,NaN,2025-03-27 18:16:11.945041,2025-03-27 18:16:15.824794,0 days 00:00:03.879753,NaN,swish,...,0.008863,NaN,NaN,AdamW,NaN,None,NaN,NaN,0.000027,FAIL
7,7,NaN,NaN,NaN,NaN,2025-03-27 18:23:23.009538,2025-03-27 18:23:23.443421,0 days 00:00:00.433883,NaN,gelu,...,0.000805,0.878234,False,SGD,NaN,None,NaN,NaN,0.000051,PRUNED
8,8,NaN,NaN,NaN,NaN,2025-03-27 18:23:23.476311,2025-03-27 18:23:59.817810,0 days 00:00:36.341499,20.0,gelu,...,0.000027,0.945208,False,SGD,NaN,Cosine,NaN,NaN,0.000559,FAIL
9,9,NaN,NaN,NaN,NaN,2025-03-27 18:26:21.707300,2025-03-27 18:26:22.319394,0 days 00:00:00.612094,NaN,swish,...,0.003359,NaN,NaN,SGD,NaN,Step,13.0,NaN,0.002863,PRUNED


## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)